# Pipeline de Validation Topologique de Plans Architecturaux
## Détection d'Hallucinations Structurelles par Analyse de Morse-Smale & Homologie Persistante

---

### Motivation & Contrainte Scientifique

Les modèles génératifs d'images (diffusion, GAN) produisent des plans architecturaux qui peuvent contenir des **hallucinations structurelles** : pièces emmurées sans issue, corridors bouchés, espaces topologiquement déconnectés. La validation manuelle est coûteuse ; une validation automatique robuste est nécessaire.

**Pourquoi ne pas appliquer l'Homologie Cubique directement sur la grille de pixels ?**

Pour une image $n \times n$ en haute résolution (ex: $1024 \times 1024$), la construction du complexe cubique génère $O(n^2)$ cubes, et le calcul de la réduction de la matrice de bords atteint une complexité temporelle $O(n^3)$ en pire cas. Pour $n = 1024$, cela représente $\approx 10^9$ opérations élémentaires — **prohibitif**.

**Notre approche : réduction via un champ scalaire continu (EDT → Morse-Smale Graph)**

Nous remplaçons la grille par un **graphe topologique de Reeb/Morse-Smale** dont la taille est $O(k)$ où $k$ est le nombre de pièces (typiquement $k \ll n^2$). L'homologie persistante de ce graphe compact capture fidèlement la topologie globale du plan.

```
Image (n²) → EDT scalaire → Bassins Morse-Smale → Graphe (k nœuds) → Filtration gudhi → β₀
```

---

### Structure du Notebook

| Bloc | Titre | Opération Clé |
|------|-------|---------------|
| 1 | Ingénierie de la Donnée | Génération d'un plan test avec hallucination |
| 2 | Champ Scalaire Continu | Transformée de Distance Euclidienne (EDT) |
| 3 | Réduction Dimensionnelle | Approximation Morse-Smale → Graphe NetworkX |
| 4 | Décision Topologique | Filtration Simpliciale & Homologie Persistante (gudhi) |

In [1]:
# ============================================================
# IMPORTS & CONFIGURATION GLOBALE
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import matplotlib.gridspec as gridspec

from scipy.ndimage import distance_transform_edt
from skimage.segmentation import watershed
from skimage.feature import peak_local_max

import networkx as nx
import gudhi

# Style global
plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#444466',
    'axes.labelcolor':  '#e0e0ff',
    'xtick.color':      '#888899',
    'ytick.color':      '#888899',
    'text.color':       '#e0e0ff',
    'grid.color':       '#2a2a4a',
    'grid.alpha':       0.5,
    'font.family':      'monospace',
})

print(f"gudhi version  : {gudhi.__version__}")
print(f"networkx version: {nx.__version__}")
print("✅ Environnement initialisé.")

ModuleNotFoundError: No module named 'numpy'

---
## Bloc 1 — Ingénierie de la Donnée : Génération du Plan Test

### Formalisation mathématique

Un plan architectural est représenté par une **image binaire** $\mathcal{I} : \mathbb{Z}^2 \to \{0, 1\}$ où :
- $\mathcal{I}(p) = 0$ : pixel **mur** (obstacle)
- $\mathcal{I}(p) = 1$ : pixel **espace libre** (espace navigable)

Nous générons un plan $512 \times 512$ contenant :
1. **Corridor central** (espace libre reliant plusieurs pièces)
2. **Pièce A** — connectée au corridor via une porte
3. **Pièce B** — connectée au corridor via une porte
4. **Pièce C** — *hallucination* : entièrement emmurée, **aucune porte**

Une porte est modélisée comme une rupture dans le mur (pixel $= 1$ sur la frontière d'une pièce).

In [ ]:
# ============================================================
# BLOC 1 : GÉNÉRATION DU PLAN ARCHITECTURAL TEST
# ============================================================

def generate_floor_plan(H: int = 512, W: int = 512) -> np.ndarray:
    """
    Génère un plan architectural binaire synthétique.
    
    Retourne un tableau uint8 (H×W) :
      - 0 : mur
      - 1 : espace libre
    
    Inclut délibérément une pièce emmurée (hallucination).
    """
    img = np.zeros((H, W), dtype=np.uint8)

    def fill_room(r0, r1, c0, c1):
        """Remplit l'intérieur d'une pièce (sans les murs)."""
        img[r0+1:r1, c0+1:c1] = 1

    def add_door(r, c, width: int = 4, axis: str = 'h'):
        """Perce une ouverture (porte) dans un mur."""
        if axis == 'h':  # porte horizontale
            img[r, c:c+width] = 1
        else:            # porte verticale
            img[r:r+width, c] = 1

    # --- Corridor central (horizontal) ---
    img[H//2 - 12 : H//2 + 12, 30 : W - 30] = 1

    # --- Pièce A (haut-gauche) : connectée ---
    fill_room(30, H//2 - 12, 30, W//3)
    add_door(H//2 - 12, W//6 - 2, width=8, axis='h')   # porte sur mur sud

    # --- Pièce B (haut-droite) : connectée ---
    fill_room(30, H//2 - 12, W//3 + 20, W - 30)
    add_door(H//2 - 12, 2*W//3 + 10, width=8, axis='h') # porte sur mur sud

    # --- Pièce D (bas-gauche) : connectée ---
    fill_room(H//2 + 12, H - 30, 30, W//2 - 20)
    add_door(H//2 + 12, W//4 - 2, width=8, axis='h')    # porte sur mur nord

    # --- Pièce C (bas-droite) : EMMURÉE — HALLUCINATION ---
    # Aucune porte n'est percée — composante topologiquement isolée
    fill_room(H//2 + 12, H - 30, W//2 + 20, W - 30)
    # ⚠️  Pas de add_door() ici : c'est la hallucination à détecter

    return img


H, W = 512, 512
floor_plan = generate_floor_plan(H, W)

# --- Visualisation Bloc 1 ---
fig, ax = plt.subplots(figsize=(7, 7))
cmap_plan = ListedColormap(['#1a1a2e', '#c8c8ff'])
ax.imshow(floor_plan, cmap=cmap_plan, interpolation='nearest')
ax.set_title("Bloc 1 — Plan Binaire (Murs=0 ■, Espace=1 □)", pad=12, fontsize=13)
ax.set_xlabel("Colonnes (pixels)")
ax.set_ylabel("Lignes (pixels)")

# Annotations des pièces
annotations = [
    (80,  W//6,    "Pièce A\n(connectée)",  '#00ff99'),
    (80,  2*W//3,  "Pièce B\n(connectée)",  '#00ff99'),
    (H-80, W//4,   "Pièce D\n(connectée)",  '#00ff99'),
    (H-80, 3*W//4, "Pièce C\n⚠ EMMURÉE",   '#ff4444'),
]
for r, c, label, color in annotations:
    ax.text(c, r, label, ha='center', va='center', fontsize=9,
            color=color, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#000022', alpha=0.7, edgecolor=color))

free_pct = 100 * floor_plan.sum() / floor_plan.size
ax.text(5, H-10, f"Pixels libres : {free_pct:.1f}%  |  Résolution : {H}×{W}",
        fontsize=8, color='#888899')

plt.tight_layout()
plt.savefig('/home/claude/bloc1_plan.png', dpi=120, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print(f"Plan généré : {H}×{W} pixels | {floor_plan.sum():,} pixels libres ({free_pct:.1f}%)")

---
## Bloc 2 — Champ Scalaire Continu : Transformée de Distance Euclidienne (EDT)

### Fondement mathématique

Soit $\mathcal{W} = \{p \in \mathbb{Z}^2 \mid \mathcal{I}(p) = 0\}$ l'ensemble des pixels murs. La **Transformée de Distance Euclidienne** est la fonction :

$$\text{EDT}(p) = \min_{q \in \mathcal{W}} \|p - q\|_2 \quad \forall p \text{ tel que } \mathcal{I}(p) = 1$$
$$\text{EDT}(p) = 0 \quad \forall p \in \mathcal{W}$$

Cette fonction transforme le masque binaire en une **surface scalaire continue** $f : \mathbb{R}^2 \to \mathbb{R}^+$ avec des propriétés topologiques remarquables :

| Structure de $\mathcal{I}$ | Structure de $f = \text{EDT}$ |
|---------------------------|-------------------------------|
| Mur (obstacle) | Zéro : $f = 0$ |
| Centre d'une pièce | **Maximum local** (plus éloigné des murs) |
| Passage / porte | **Point-selle** (col entre deux maxima) |
| Couloir étroit | Crête de faible amplitude |

### Lien avec la Théorie de Morse

La fonction EDT est une approximation discrète d'une **fonction de Morse** sur la variété 2D représentant l'espace libre. Ses points critiques (maxima, minima, points-selles) définissent la **décomposition de Morse-Smale** utilisée au Bloc 3. Notamment :

$$\text{Largeur d'une porte} \approx 2 \times \text{EDT}(\text{col}_{\text{selle}})$$

L'algorithme scipy utilise une méthode de scan en deux passes d'ordre $O(n^2)$, bien plus efficace que l'approche naïve $O(n^4)$.

In [ ]:
# ============================================================
# BLOC 2 : CALCUL DE LA TRANSFORMÉE DE DISTANCE EUCLIDIENNE
# ============================================================

print("⏳ Calcul de l'EDT...")
edt = distance_transform_edt(floor_plan).astype(np.float32)
print(f"✅ EDT calculée | max={edt.max():.2f}px | min={edt[floor_plan==1].min():.2f}px")
print(f"   Interprétation : max = rayon de la plus grande sphère inscrite ≈ {edt.max():.1f}px")

# --- Visualisation Bloc 2 ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Vue 2D de l'EDT (heatmap)
im = axes[0].imshow(edt, cmap='plasma', interpolation='bilinear', origin='upper')
plt.colorbar(im, ax=axes[0], label='Distance aux murs (pixels)', fraction=0.046, pad=0.04)
axes[0].set_title("Champ EDT — Vue 2D (Heatmap)", fontsize=12)
axes[0].set_xlabel("Colonnes")
axes[0].set_ylabel("Lignes")

# Superposition masque + contours EDT
axes[1].imshow(floor_plan, cmap=ListedColormap(['#0a0a1a', '#2a2a4a']), alpha=0.8)
contour_levels = np.linspace(edt[edt>0].min(), edt.max(), 18)
cs = axes[1].contour(edt, levels=contour_levels, cmap='plasma', linewidths=0.8, alpha=0.9)
axes[1].set_title("Lignes de Niveau EDT (Approximation Morse)", fontsize=12)
axes[1].set_xlabel("Colonnes")
axes[1].set_ylabel("Lignes")

# Annotations topologiques
axes[1].text(W//2, H//2 + 5, "Corridor\n(crête)",
             ha='center', va='center', fontsize=8, color='#ffff88',
             bbox=dict(facecolor='#000033', alpha=0.6, edgecolor='#ffff88', boxstyle='round'))

fig.suptitle("Bloc 2 — EDT : Surface Scalaire Continue sur l'Espace Libre",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/home/claude/bloc2_edt.png', dpi=120, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

---
## Bloc 3 — Réduction Dimensionnelle : Graphe de Morse-Smale

### Théorie

La **décomposition de Morse-Smale** d'une fonction $f : M \to \mathbb{R}$ partition la variété $M$ en **bassins d'attraction** : les cellules de descente gradient (stable manifolds) partant des maxima. Deux bassins adjacents sont séparés par un **col de selle** (saddle point).

Nous approximons cette décomposition en 3 étapes :

#### Étape 3.1 — Extraction des Maxima Locaux (Nœuds)

Un pixel $p^*$ est un maximum local de l'EDT si :
$$\text{EDT}(p^*) \geq \text{EDT}(q) \quad \forall q \in \mathcal{N}(p^*, r)$$

où $\mathcal{N}(p^*, r)$ est le voisinage de rayon $r$. Ces maxima correspondent aux **centroïdes géométriques** des pièces.

#### Étape 3.2 — Segmentation par Ligne de Partage des Eaux (LPE / Watershed)

L'algorithme Watershed appliqué sur $-\text{EDT}$ segmente l'espace libre en **bassins de Voronoï pondérés** autour de chaque maximum. La LPE est l'approximation discrète la plus robuste des lignes de flot de Morse-Smale.

$$\text{Basin}(p^*_i) = \{p \mid \text{ligne de descente de } p \text{ converge vers } p^*_i\}$$

#### Étape 3.3 — Extraction des Points-Selles (Pondération des Arêtes)

Le **col** entre deux bassins $B_i$ et $B_j$ est le point frontière $s_{ij}$ maximisant l'EDT sur leur frontière commune :

$$\text{saddle}(i,j) = \max_{p \in \partial B_i \cap \partial B_j} \text{EDT}(p)$$

Cette valeur encode la **largeur de la porte** séparant les deux pièces ($\approx 2 \times \text{saddle}(i,j)$ pixels).

#### Étape 3.4 — Construction du Graphe Morse-Smale

$$G_{MS} = (V, E, w) \quad \text{où} \quad V = \{p^*_i\},\; E = \{(i,j) \mid \partial B_i \cap \partial B_j \neq \emptyset\},\; w(i,j) = \text{saddle}(i,j)$$

**Réduction de complexité** : $|V| \ll n^2$ et $|E| \ll n^4$, typiquement $|V| \sim O(k)$ pour $k$ pièces.

In [ ]:
# ============================================================
# BLOC 3 — ÉTAPE 3.1 : EXTRACTION DES MAXIMA LOCAUX (NŒUDS)
# ============================================================

MIN_DISTANCE_PX  = 15    # séparation minimale entre maxima (évite les doublons)
MIN_EDT_ALTITUDE = 5.0   # seuil de bruit : ignore les petits recoins

# peak_local_max retourne les coordonnées (row, col) des maxima locaux
maxima_coords = peak_local_max(
    edt,
    min_distance=MIN_DISTANCE_PX,
    threshold_abs=MIN_EDT_ALTITUDE,
    exclude_border=True
)

n_nodes = len(maxima_coords)
print(f"Étape 3.1 | Maxima locaux trouvés : {n_nodes}")
print(f"  Altitudes EDT : min={edt[maxima_coords[:,0], maxima_coords[:,1]].min():.1f}px "
      f"| max={edt[maxima_coords[:,0], maxima_coords[:,1]].max():.1f}px")

In [ ]:
# ============================================================
# BLOC 3 — ÉTAPE 3.2 : WATERSHED (LIGNE DE PARTAGE DES EAUX)
# ============================================================

# Création de l'image markers : chaque maximum est un germe de bassin
markers = np.zeros((H, W), dtype=np.int32)
for idx, (r, c) in enumerate(maxima_coords, start=1):
    markers[r, c] = idx

# Watershed sur -EDT : les maxima de l'EDT deviennent des minima (bassins)
# mask=floor_plan.astype(bool) : la LPE reste confinée à l'espace libre
basin_labels = watershed(
    image=-edt,
    markers=markers,
    mask=floor_plan.astype(bool)
)

n_basins = int(basin_labels.max())
print(f"Étape 3.2 | Bassins de Morse-Smale : {n_basins}")
assert n_basins == n_nodes, "Incohérence : #bassins ≠ #maxima"

# Visualisation intermédiaire des bassins
fig, ax = plt.subplots(figsize=(8, 8))
# Masque fond
ax.imshow(floor_plan, cmap=ListedColormap(['#080810', '#111128']), alpha=1.0)
# Bassins colorisés
basin_viz = np.ma.masked_where(basin_labels == 0, basin_labels)
ax.imshow(basin_viz, cmap='tab20', alpha=0.65, interpolation='nearest')
# Frontières des bassins
ax.contour(basin_labels, levels=np.arange(0.5, n_basins+1), colors='white',
           linewidths=0.4, alpha=0.5)
# Maxima
ax.scatter(maxima_coords[:,1], maxima_coords[:,0],
           c='#ffdd00', s=60, zorder=5, marker='*',
           edgecolors='#ff8800', linewidths=0.8, label='Maxima EDT (nœuds)')

ax.set_title(f"Étape 3.2 — Décomposition Watershed ({n_basins} bassins)", fontsize=12)
ax.legend(loc='lower right', fontsize=9)
ax.set_xlabel("Colonnes"); ax.set_ylabel("Lignes")
plt.tight_layout()
plt.savefig('/home/claude/bloc3a_watershed.png', dpi=120, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

In [ ]:
# ============================================================
# BLOC 3 — ÉTAPE 3.3 : EXTRACTION DES POINTS-SELLES (COLS)
# ============================================================

def extract_saddle_heights(
    labels: np.ndarray,
    edt:    np.ndarray,
    mask:   np.ndarray
) -> dict:
    """
    Parcourt tous les pixels libres et détecte les frontières inter-bassins.
    Pour chaque paire adjacente de bassins (i, j), conserve l'altitude EDT
    maximale sur leur frontière commune (= largeur du col / porte).
    
    Complexité : O(|espace libre|) = O(n²) — linéaire en taille d'image.
    
    Retourne : dict[(basin_i, basin_j)] -> saddle_height (float)
    """
    saddles = {}  # (min_basin, max_basin) -> max_edt_at_boundary
    
    rows, cols = np.where(mask > 0)
    
    # Voisinage 4-connexe (on peut étendre à 8 pour plus de robustesse)
    neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    
    H_img, W_img = labels.shape
    for r, c in zip(rows, cols):
        b_current = int(labels[r, c])
        if b_current == 0:
            continue
        for dr, dc in neighbors:
            nr, nc = r + dr, c + dc
            if not (0 <= nr < H_img and 0 <= nc < W_img):
                continue
            b_neighbor = int(labels[nr, nc])
            if b_neighbor > 0 and b_neighbor != b_current:
                key = (min(b_current, b_neighbor), max(b_current, b_neighbor))
                # L'altitude du col = EDT au pixel courant (sur la frontière)
                val = float(edt[r, c])
                if key not in saddles or val > saddles[key]:
                    saddles[key] = val
    return saddles


print("⏳ Extraction des points-selles (scan des frontières inter-bassins)...")
saddle_dict = extract_saddle_heights(basin_labels, edt, floor_plan)
n_edges = len(saddle_dict)

saddle_vals = np.array(list(saddle_dict.values()))
print(f"✅ Étape 3.3 | Arêtes (cols / portes) extraites : {n_edges}")
print(f"   Altitude de selle : min={saddle_vals.min():.2f}px | "
      f"max={saddle_vals.max():.2f}px | mean={saddle_vals.mean():.2f}px")

In [ ]:
# ============================================================
# BLOC 3 — ÉTAPE 3.4 : CONSTRUCTION DU GRAPHE MORSE-SMALE
# ============================================================

G = nx.Graph()

# Ajout des nœuds avec attributs géométriques et topologiques
for idx, (r, c) in enumerate(maxima_coords, start=1):
    G.add_node(
        idx,
        pos     = (int(c), int(r)),      # position image (col, row)
        edt_val = float(edt[r, c]),      # altitude = "rayon" de la pièce
        basin_id= idx
    )

# Ajout des arêtes pondérées par l'altitude du col
for (u, v), saddle_h in saddle_dict.items():
    G.add_edge(u, v, weight=saddle_h, saddle=saddle_h)

print(f"Graphe Morse-Smale G = (V={G.number_of_nodes()}, E={G.number_of_edges()})")
print(f"  Composantes connexes : {nx.number_connected_components(G)}")
print(f"  Réduction dimensionnelle : {H*W:,} pixels → {G.number_of_nodes()} nœuds "
      f"({100*G.number_of_nodes()/(H*W):.4f}% de la grille)")

# Composantes connexes : on s'attend à 2 (pièces connectées + pièce emmurée)
components = list(nx.connected_components(G))
print(f"\n  Détail des composantes :")
for i, comp in enumerate(components, 1):
    print(f"    Composante {i} : {len(comp)} nœud(s) → {comp}")

In [ ]:
# ============================================================
# BLOC 3 — VISUALISATION FINALE : GRAPHE SUPERPOSÉ AU PLAN
# ============================================================

fig, ax = plt.subplots(figsize=(10, 10))

# Fond : EDT
ax.imshow(edt, cmap='inferno', alpha=0.5, interpolation='bilinear')
ax.imshow(1 - floor_plan, cmap=ListedColormap(['#00000000', '#1a1a3a']),
          alpha=0.7, interpolation='nearest')

# Extraction positions pour NetworkX
pos_dict = {n: data['pos'] for n, data in G.nodes(data=True)}
edt_vals = np.array([data['edt_val'] for _, data in G.nodes(data=True)])
edge_weights = [d['weight'] for _, _, d in G.edges(data=True)]

# Coloration des nœuds par composante connexe
comp_color_map = {}
comp_colors = ['#00ff99', '#ff4444', '#44aaff', '#ffaa00', '#ff44ff']
for ci, comp in enumerate(components):
    is_hallucination = (len(comp) == 1 and 
                        not any(G.degree(n) > 0 for n in comp))
    # Identifie la composante isolée comme hallucination potentielle
    color = '#ff4444' if len(comp) < len(components[0]) else '#00ff99'
    if len(components) == 1:
        color = '#00ff99'
    for n in comp:
        comp_color_map[n] = color

node_colors = [comp_color_map[n] for n in G.nodes()]

# Taille des nœuds proportionnelle à l'altitude EDT (rayon de la pièce)
node_sizes = [max(50, 8 * data['edt_val']**1.2) for _, data in G.nodes(data=True)]

# Arêtes colorisées par poids (largeur du col)
if edge_weights:
    ew_norm = np.array(edge_weights)
    ew_norm = (ew_norm - ew_norm.min()) / (ew_norm.max() - ew_norm.min() + 1e-6)
    edge_colors = plt.cm.YlOrRd(ew_norm)
    edge_widths = [1.5 + 3.0 * w for w in ew_norm]
else:
    edge_colors = 'white'
    edge_widths = 2.0

nx.draw_networkx_edges(
    G, pos_dict, ax=ax,
    edge_color=edge_colors,
    width=edge_widths,
    alpha=0.85
)
nx.draw_networkx_nodes(
    G, pos_dict, ax=ax,
    node_color=node_colors,
    node_size=node_sizes,
    edgecolors='white',
    linewidths=1.2
)
nx.draw_networkx_labels(
    G, pos_dict, ax=ax,
    labels={n: str(n) for n in G.nodes()},
    font_size=7, font_color='white'
)

# Légende
patches = [
    mpatches.Patch(color='#00ff99', label='Nœud connecté (pièce accessible)'),
    mpatches.Patch(color='#ff4444', label='Nœud isolé ⚠ Hallucination suspectée'),
    mpatches.Patch(color='#ffaa00', label='Arête = col/porte (pondération EDT)'),
]
ax.legend(handles=patches, loc='lower left', fontsize=9,
          facecolor='#0a0a20', edgecolor='#444466')

n_comp = nx.number_connected_components(G)
ax.set_title(
    f"Bloc 3 — Graphe Morse-Smale G = (V={G.number_of_nodes()}, E={G.number_of_edges()}) "
    f"| {n_comp} composante(s) connexe(s)",
    fontsize=12, pad=12
)
ax.set_xlabel("Colonnes (pixels)")
ax.set_ylabel("Lignes (pixels)")
ax.set_xlim(0, W); ax.set_ylim(H, 0)

plt.tight_layout()
plt.savefig('/home/claude/bloc3b_graph.png', dpi=120, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print(f"Graphe sauvegardé. Réduction : {H}×{W} = {H*W:,} → {G.number_of_nodes()} nœuds")

---
## Bloc 4 — Décision Topologique : Filtration Simpliciale & Homologie Persistante

### Théorie de la Filtration

#### 4.1 Construction du Complexe Simplicial Filtré

Un **complexe simplicial** $K$ est construit à partir du graphe $G_{MS}$ :
- **0-simplexes** (nœuds) : chaque pièce $v_i$, insérée à la filtration $f(v_i) = -\text{EDT}(v_i)$ *(nœuds à haute altitude EDT naissent en premier dans la filtration décroissante)*
- **1-simplexes** (arêtes) : chaque connexion $(v_i, v_j)$, insérée à $f(v_i, v_j) = -\text{saddle}(i,j)$ *(arêtes à selle haute = portes larges mergent tôt)*

#### 4.2 Filtration Décroissante (Super-levelset)

On parcourt les niveaux $t$ de $+\infty$ à $-\infty$ (ou équivalent, de l'EDT max à 0) :

$$K_t = \{\sigma \in K \mid f(\sigma) \leq t\}$$

À chaque seuil $t$ :
- Si un nœud isolé apparaît → **naissance** d'une composante $H_0$
- Si une arête connecte deux composantes → **mort** de la composante jeune (Elder Rule)

#### 4.3 Homologie Persistante $H_0$ et Interprétation

Le module de persistance $H_0$ encode la **connectivité** du plan :

$$\text{PD}_0 = \{(b_i, d_i)\} \quad \text{où} \quad d_i - b_i = \text{persistance de la composante } i$$

| Paire $(b, d)$ | Interprétation architecturale |
|----------------|-------------------------------|
| $(b, +\infty)$ | Composante connexe **qui ne meurt jamais** = espace non connecté au reste |
| $(b, d)$, $d < 0$ | Composante mergée avec une autre via une porte à altitude $|d|$ |

**Règle de validation :**

$$\beta_0 = \#\{(b, d) \in \text{PD}_0 \mid d = +\infty\}$$

- $\beta_0 = 1$ : plan connexe → **VALIDE**
- $\beta_0 > 1$ : présence de $\beta_0 - 1$ composantes isolées → **HALLUCINATION DÉTECTÉE**

In [ ]:
# ============================================================
# BLOC 4 — FILTRATION SIMPLICIALE & CALCUL DE PERSISTANCE
# ============================================================

# --- Construction du SimplexTree gudhi ---
st = gudhi.SimplexTree()

# Insertion des 0-simplexes (nœuds)
# Valeur de filtration : -EDT (filtration décroissante → nœuds à haute EDT naissent tôt)
for node, data in G.nodes(data=True):
    st.insert([node], filtration=-data['edt_val'])

# Insertion des 1-simplexes (arêtes)
# Valeur de filtration : -saddle (portes larges = merge tôt)
for u, v, data in G.edges(data=True):
    f_edge = -data['saddle']
    st.insert([u, v], filtration=f_edge)

print(f"SimplexTree initialisé :")
print(f"  Nombre de simplexes : {st.num_simplices()}")
print(f"  Dimension max       : {st.dimension()}")
print(f"  Filtration range    : [{st.filtration(min(G.nodes())):.2f}, "  
      f"max à calculer]")

# --- Calcul de l'Homologie Persistante ---
st.compute_persistence(homology_coeff_field=2, min_persistence=0.0)

all_pairs = st.persistence()
print(f"\nPaires de persistance calculées : {len(all_pairs)}")

# Extraction des paires H0
h0_pairs = [(b, d) for (dim, (b, d)) in all_pairs if dim == 0]
h0_infinite = [(b, d) for (b, d) in h0_pairs if d == float('inf')]
h0_finite   = [(b, d) for (b, d) in h0_pairs if d != float('inf')]

print(f"\nH₀ — Composantes Connexes :")
print(f"  Paires infinies (β₀) : {len(h0_infinite)} → composantes permanentes")
print(f"  Paires finies        : {len(h0_finite)}  → merges (portes)")
print()
print(f"  Détail paires infinies :")
for b, d in sorted(h0_infinite):
    print(f"    birth={b:.3f}  →  death=+∞  "
          f"(nœud d'altitude EDT ≈ {-b:.1f}px — JAMAIS connecté)")

print(f"\n  Détail merges (portes) : {len(h0_finite)} événements")
for b, d in sorted(h0_finite, key=lambda x: x[1])[:5]:
    print(f"    birth={b:.3f}  death={d:.3f}  "
          f"(porte de largeur ≈ {-d*2:.1f}px à altitude EDT={-d:.1f}px)")

In [ ]:
# ============================================================
# BLOC 4 — VISUALISATION : DIAGRAMME DE PERSISTANCE & BARCODE
# ============================================================

NOISE_THRESHOLD = -MIN_EDT_ALTITUDE  # seuil en filtration (= -altitude)
INF_PROXY       = 1.2 * max(abs(b) for b, _ in h0_pairs + [(0,0)])

fig = plt.figure(figsize=(16, 7))
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

# --- Sous-figure gauche : Diagramme de Persistance H₀ ---
ax_pd = fig.add_subplot(gs[0])

# Paires finies (merges / portes)
for b, d in h0_finite:
    ax_pd.scatter(b, d, c='#44aaff', s=60, zorder=3, alpha=0.8)

# Paires infinies (composantes isolées) → projetées sur la droite y=INF_PROXY
for b, d in h0_infinite:
    color = '#00ff99' if len(h0_infinite) == 1 else '#ff4444'
    ax_pd.scatter(b, INF_PROXY, c=color, s=120, zorder=5,
                  marker='D', edgecolors='white', linewidths=1.2)
    ax_pd.annotate(f"β₀\nbirth={b:.1f}",
                   xy=(b, INF_PROXY), xytext=(b - 2, INF_PROXY * 0.82),
                   fontsize=8, color=color, ha='center',
                   arrowprops=dict(arrowstyle='->', color=color, lw=1.2))

# Diagonale
all_b = [b for b,_ in h0_pairs]
all_d = [d if d != float('inf') else INF_PROXY for _,d in h0_pairs]
lim_min = min(all_b + all_d) - 1
lim_max = max(all_b + all_d) + 1
ax_pd.plot([lim_min, lim_max], [lim_min, lim_max],
           'w--', alpha=0.3, linewidth=1, label='Diagonale (pers.=0)')

# Seuil de bruit
ax_pd.axhline(NOISE_THRESHOLD, color='#ffaa00', linewidth=1.2,
              linestyle=':', alpha=0.8, label=f'Seuil bruit = {NOISE_THRESHOLD:.1f}')
ax_pd.axhline(INF_PROXY, color='#888899', linewidth=0.8,
              linestyle='--', alpha=0.5, label='Proxy +∞')
ax_pd.text(lim_min + 0.5, INF_PROXY + 0.3, '+∞ (inf)', fontsize=8, color='#888899')

ax_pd.set_xlabel("Naissance (birth)", fontsize=11)
ax_pd.set_ylabel("Mort (death)", fontsize=11)
ax_pd.set_title(f"Diagramme de Persistance H₀\n(β₀ = {len(h0_infinite)} composante(s) permanente(s))",
                fontsize=12)
ax_pd.legend(fontsize=8)
ax_pd.grid(True, alpha=0.3)

# --- Sous-figure droite : Barcode H₀ ---
ax_bc = fig.add_subplot(gs[1])

sorted_pairs = sorted(h0_finite + [(b, INF_PROXY * 1.05) for b, _ in h0_infinite],
                      key=lambda x: x[0])

for y_pos, (b, d) in enumerate(sorted_pairs):
    is_inf = (b, float('inf')) in h0_infinite
    color  = '#ff4444' if is_inf and len(h0_infinite) > 1 else (
             '#00ff99' if is_inf else '#44aaff')
    lw = 3.5 if is_inf else 2.0
    ax_bc.plot([b, d], [y_pos, y_pos], color=color, linewidth=lw, solid_capstyle='butt')
    if is_inf:
        ax_bc.annotate('→ +∞', xy=(d, y_pos), fontsize=8, color=color,
                       va='center', ha='left')

ax_bc.axvline(NOISE_THRESHOLD, color='#ffaa00', linewidth=1.2,
              linestyle=':', label=f'Seuil bruit')

ax_bc.set_xlabel("Valeur de filtration", fontsize=11)
ax_bc.set_ylabel("Composantes H₀", fontsize=11)
ax_bc.set_title("Code-Barres (Barcode) H₀", fontsize=12)
ax_bc.set_yticks(range(len(sorted_pairs)))
ax_bc.set_yticklabels([f"Comp. {i+1}" for i in range(len(sorted_pairs))], fontsize=8)
ax_bc.legend(fontsize=9)
ax_bc.grid(True, alpha=0.3, axis='x')

# Légende globale
from matplotlib.lines import Line2D
legend_els = [
    Line2D([0],[0], color='#ff4444', lw=3, label='Composante isolée (hallucination)'),
    Line2D([0],[0], color='#00ff99', lw=3, label='Composante principale (valide)'),
    Line2D([0],[0], color='#44aaff', lw=2, label='Merge (porte / col)'),
]
fig.legend(handles=legend_els, loc='upper center', ncol=3, fontsize=9,
           facecolor='#0f0f0f', edgecolor='#444466', bbox_to_anchor=(0.5, 0.0))

fig.suptitle("Bloc 4 — Homologie Persistante H₀ : Topologie de Connectivité du Plan",
             fontsize=14, y=1.02)
plt.savefig('/home/claude/bloc4_persistence.png', dpi=120, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

In [ ]:
# ============================================================
# BLOC 4 — DÉCISION TOPOLOGIQUE FINALE
# ============================================================

NOISE_THRESHOLD_EDT = MIN_EDT_ALTITUDE  # seuil en valeur EDT positive

beta_0 = len(h0_infinite)

# Composantes suspectées : nœuds isolés dont la naissance est au-dessus du bruit
hallucination_candidates = [
    (b, d) for (b, d) in h0_infinite
    if abs(b) > NOISE_THRESHOLD_EDT   # composante de taille non-triviale
]

print("=" * 65)
print("     RAPPORT DE VALIDATION TOPOLOGIQUE")
print("=" * 65)
print(f"  Plan analysé          : {H}×{W} pixels")
print(f"  Nœuds (pièces/zones)  : {G.number_of_nodes()}")
print(f"  Arêtes (portes/cols)  : {G.number_of_edges()}")
print(f"  β₀ (composantes H₀)   : {beta_0}")
print(f"  Seuil de bruit EDT    : {NOISE_THRESHOLD_EDT} px")
print(f"  Hallucinations signif.: {len(hallucination_candidates)}")
print("-" * 65)

if beta_0 == 1:
    print()
    print("  ✅  PLAN VALIDE")
    print("      β₀ = 1 → Toutes les pièces sont topologiquement connectées.")
    print("      Aucune hallucination structurelle détectée.")
    print()
else:
    n_hallucinations = beta_0 - 1
    print()
    print("  🔴  PLAN REJETÉ : HALLUCINATION DÉTECTÉE")
    print(f"      β₀ = {beta_0} → {n_hallucinations} composante(s) emmurée(s) détectée(s).")
    print(f"      Pièce(s) sans accès : {n_hallucinations} zone(s) topologiquement isolée(s).")
    print()
    print("  Détail des composantes isolées :")
    for i, (b, d) in enumerate(sorted(h0_infinite, key=lambda x: x[0]), 1):
        edt_radius = abs(b)
        print(f"    [{i}] Rayon EDT ≈ {edt_radius:.1f}px "
              f"(espace ≈ {int(np.pi * edt_radius**2)} px²) "
              f"— jamais connectée au reste du plan")
    print()
    print("  ► Action recommandée : Régénérer le plan ou ajouter une porte")
    print(f"    à toute pièce dont l'EDT dépasse {NOISE_THRESHOLD_EDT}px.")

print("=" * 65)

In [ ]:
# ============================================================
# RÉCAPITULATIF VISUEL : 4 BLOCS EN UNE FIGURE
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 16))
fig.suptitle("Pipeline de Validation Topologique — Vue Synthétique",
             fontsize=16, y=0.98)

# B1: Plan binaire
axes[0,0].imshow(floor_plan, cmap=ListedColormap(['#1a1a2e', '#c8c8ff']),
                 interpolation='nearest')
axes[0,0].set_title("Bloc 1 — Plan Binaire", fontsize=12)
for r, c, lbl, col in annotations:
    axes[0,0].text(c, r, lbl, ha='center', va='center', fontsize=8,
                   color=col, fontweight='bold',
                   bbox=dict(boxstyle='round,pad=0.2', facecolor='#000022', alpha=0.7,
                             edgecolor=col))

# B2: EDT
im2 = axes[0,1].imshow(edt, cmap='plasma', interpolation='bilinear')
plt.colorbar(im2, ax=axes[0,1], fraction=0.046, pad=0.04, label='EDT (px)')
axes[0,1].set_title("Bloc 2 — Champ EDT", fontsize=12)

# B3: Graphe
axes[1,0].imshow(edt, cmap='inferno', alpha=0.4, interpolation='bilinear')
axes[1,0].imshow(1 - floor_plan, cmap=ListedColormap(['#00000000','#1a1a3a']),
                 alpha=0.7, interpolation='nearest')
if edge_weights:
    nx.draw_networkx_edges(G, pos_dict, ax=axes[1,0],
                           edge_color=edge_colors, width=edge_widths, alpha=0.9)
nx.draw_networkx_nodes(G, pos_dict, ax=axes[1,0],
                       node_color=node_colors, node_size=node_sizes,
                       edgecolors='white', linewidths=1.0)
axes[1,0].set_xlim(0, W); axes[1,0].set_ylim(H, 0)
axes[1,0].set_title(f"Bloc 3 — Graphe Morse-Smale (V={G.number_of_nodes()}, "
                    f"E={G.number_of_edges()})", fontsize=12)

# B4: Diagramme de persistance simplifié
ax4 = axes[1,1]
for b, d in h0_finite:
    ax4.scatter(b, d, c='#44aaff', s=55, zorder=3, alpha=0.85)
for b, d in h0_infinite:
    c4 = '#00ff99' if beta_0 == 1 else '#ff4444'
    ax4.scatter(b, INF_PROXY, c=c4, s=150, marker='D',
                edgecolors='white', linewidths=1.5, zorder=5)
ax4.plot([lim_min, lim_max], [lim_min, lim_max], 'w--', alpha=0.25, lw=1)
ax4.axhline(INF_PROXY, color='#888899', lw=0.8, ls='--', alpha=0.5)
ax4.text(lim_min + 0.5, INF_PROXY + 0.2, '+∞', fontsize=9, color='#888899')
ax4.set_xlabel("birth"); ax4.set_ylabel("death")
verdict = "✅ VALIDE" if beta_0 == 1 else f"🔴 REJETÉ (β₀={beta_0})"
ax4.set_title(f"Bloc 4 — Persistance H₀ | {verdict}", fontsize=12)
ax4.grid(True, alpha=0.3)

for ax in axes.flat:
    ax.set_xlabel(ax.get_xlabel() or "Colonnes")
    ax.set_ylabel(ax.get_ylabel() or "Lignes")

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('/home/claude/pipeline_synthese.png', dpi=120, bbox_inches='tight',
            facecolor='#0f0f0f')
plt.show()
print("Figure de synthèse sauvegardée : pipeline_synthese.png")

---
## Conclusion & Extensions

### Récapitulatif du Pipeline

```
Plan IA (512×512)
    │
    ▼ Seuillage binaire
Masque Murs/Espace (262 144 pixels)
    │
    ▼ scipy.ndimage.distance_transform_edt  [O(n²)]
Champ EDT scalaire — Encode largeur des portes via points-selles
    │
    ▼ skimage.peak_local_max + watershed   [O(n²)]
Décomposition Morse-Smale → k bassins (~20-50 nœuds)
    │
    ▼ Scan frontières inter-bassins        [O(n²)]
Graphe NetworkX pondéré G = (V=k, E≤k²)
    │
    ▼ gudhi.SimplexTree + compute_persistence [O(k^ω) ≪ O(n³)]
Diagramme de Persistance H₀
    │
    ▼ Condition β₀
VALIDE ou REJETÉ (Hallucination)
```

### Complexité Totale

| Étape | Complexité | Commentaire |
|-------|-----------|-------------|
| EDT | $O(n^2)$ | 2 passes linéaires |
| Watershed | $O(n^2 \log n)$ | File de priorité |
| Extraction selles | $O(n^2)$ | Scan 4-connexe |
| Filtration gudhi | $O(k^\omega)$ | $k \ll n$, $\omega \approx 2.37$ |
| **Total** | $O(n^2 \log n)$ | **vs $O(n^3)$ naïf** |

### Extensions Possibles

1. **H₁ (Boucles)** : Injecter des 2-simplexes (triangles) pour détecter des cycles non contractiles (pièces annulaires aberrantes)
2. **Distance de Bottleneck** : Comparer le diagramme du plan généré avec celui d'un plan de référence valide
3. **Batch Pipeline** : Appliquer sur des milliers de plans générés par diffusion pour filtrage automatique
4. **Persistance de Chemin** : Pondérer les arêtes par la persistance du col pour distinguer portes étroites et murs percés
5. **Graphe de Reeb** : Alternative plus théorique, exacte sur les variétés lisses